# MAX + Mistral (souverain) → endpoint OpenAI-compatible pour Le Rapporteur

But : servir **Mistral** via **Modular MAX** sur le GPU Colab (ton Google AI One),
l'exposer en HTTP OpenAI-compatible, et brancher notre pipeline dessus
(`LLM_BASE_URL` dans `.env`, `MODE=live`).

**Avant de lancer :** menu *Exécution → Modifier le type d'exécution → GPU* (T4/L4/A100).

Honnêteté narratif : ici MAX tourne sur le GPU **Colab (NVIDIA)** — ça prouve
*Mistral live* + *MAX comme runtime portable*. Le point **non-NVIDIA** est déjà
tenu par **Qualcomm Cloud AI 100** (Cirrascale). L'objectif AMD/MAX se fait sur
AMD Dev Cloud.

## 1. Installer Modular MAX

In [ ]:
!pip -q install modular huggingface_hub
!max --version || python -c "import max; print('max importé')"

## 2. Accès au modèle Mistral (gated sur Hugging Face)
Mistral-7B-Instruct est *gated* : accepte la licence sur sa page HF, crée un
token (Settings → Access Tokens), puis colle-le ci-dessous. 
Alternative non-gated : mets `MODEL='NousResearch/Hermes-2-Pro-Mistral-7B'` (base Mistral).

In [ ]:
from huggingface_hub import login
from getpass import getpass
login(getpass('HF token: '))  # ne s'affiche pas
MODEL = 'mistralai/Mistral-7B-Instruct-v0.3'  # gated ; sinon voir alternative ci-dessus

## 3. Servir Mistral via MAX (OpenAI-compatible, port 8000)
Démarre en arrière-plan ; le 1er lancement télécharge les poids (quelques min).

In [ ]:
import subprocess, time, os
os.makedirs('logs', exist_ok=True)
srv = subprocess.Popen(['max', 'serve', '--model-path', MODEL, '--port', '8000'],
                       stdout=open('logs/max.log','w'), stderr=subprocess.STDOUT)
print('MAX serve PID', srv.pid, '— suivi: !tail -f logs/max.log')
# attendre que le endpoint réponde
import urllib.request
for i in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); print('READY'); break
    except Exception:
        time.sleep(5)
else:
    print('pas prêt — voir logs/max.log')

### (Repli fiable) vLLM si MAX coince
```python
!pip -q install vllm
srv = subprocess.Popen(['python','-m','vllm.entrypoints.openai.api_server',
                        '--model', MODEL, '--port','8000'],
                       stdout=open('logs/vllm.log','w'), stderr=subprocess.STDOUT)
```
Même interface OpenAI-compatible → rien à changer côté pipeline.

## 4. Exposer l'endpoint (tunnel public, sans compte)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, re, time
tun = subprocess.Popen(['./cloudflared','tunnel','--url','http://localhost:8000','--no-autoupdate'],
                       stdout=open('logs/tunnel.log','w'), stderr=subprocess.STDOUT)
url=None
for _ in range(30):
    time.sleep(2)
    try:
        m=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('logs/tunnel.log').read())
        if m: url=m.group(0); break
    except FileNotFoundError: pass
print('URL PUBLIQUE:', url)

## 5. Test + à mettre dans ton `.env` local
```bash
MODE=live
LLM_BASE_URL=<URL PUBLIQUE>/v1
LLM_API_KEY=sk-noauth        # MAX/vLLM local : clé factice acceptée
LLM_MODEL=mistralai/Mistral-7B-Instruct-v0.3
```
Puis, en local : `MODE=live uv run python -m src.cli "Quelle est la durée légale du travail ?"`
→ Mistral/MAX répond, vérif Canutes réelle, panneau frugalité (latence/tokens/s).

In [ ]:
import json, urllib.request
req = urllib.request.Request('http://localhost:8000/v1/chat/completions',
    data=json.dumps({'model':MODEL,'messages':[{'role':'user','content':'Réponds en 5 mots: bonjour'}],'max_tokens':30}).encode(),
    headers={'Content-Type':'application/json'})
print(json.load(urllib.request.urlopen(req))['choices'][0]['message']['content'])